# GISLR — ASL-LEX Sentences: a gloss-restricted variant of GISLR-Sentences

**What this notebook does.** Pipeline stage (dataset build), not a diagnostic. Part of **recognition_2**
(TODO §14), a workstream kept deliberately isolated from `experiments/recognition/` — nothing here edits
`sb.recognize`'s existing modules, only `sb.recognize.recognition2` (new, isolated).

Builds a **second** continuous-sentence test set, filtered to the 233/250 GISLR glosses that map into
ASL-LEX 2.0 (`sb.recognize.aslex`, gloss-mapping check this conversation 2026-09-26/27) — for phonology
work that needs ASL-LEX ground truth per sign (`gislr.1.pipeline.stage1-rules.ipynb`, TODO §14). It reuses,
**unmodified**: the committed sentence corpus (`sb.recognize.sequences.corpus`) and the landmark-space
composer GISLR-Sentences v1 already uses (`sb.recognize.sequences.compose`) — only the corpus's sentence
list and the source clip pool are filtered first, in `sb.recognize.recognition2.dataset`. Built in landmark
space from the start, the same way GISLR-Sentences v1 is, so this dataset does not carry the feature-space
composer mismatch that broke the `P1` continuous phonology model (`phonology-models.md` §9).

**Scope decision (this notebook, 2026-09-27):** only the `sentence` split is built here, not GISLR-Sentences
v1's `control` split (random-order clips, no language prior) — not needed for the phonology validation this
feeds; can be added later with `compose.plan_control` if wanted.

**Artifacts:**

| path | what |
|---|---|
| `data/cache/gislr_aslex_sentences/<version>/plan.json` | the sentence-instantiation plan, keyed by config hash |
| `data/cache/gislr_aslex_sentences/<version>/manifest.json` | per-sequence build status (resumable) |
| `data/cache/gislr_aslex_sentences/<version>/sequences/sentence/*.npz` | the built continuous sequences |

**Resumability**: manifest-driven, same pattern as `gislr.0.dataset.sentences.ipynb` — re-running this
notebook only builds sequences not already marked `done`.


## Setup

In [1]:
# ============================================================
# Imports, tunables, resolved paths
# ============================================================
import json
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sb.core.paths import CACHE_DIR, EXPERIMENTS_DIR, gislr_dir
from sb.core.vocab import load_label_map
from sb.recognize.recognition2 import dataset as aslex_dataset
from sb.recognize.sequences import compose

CONFIG_PATH = EXPERIMENTS_DIR / "recognition2" / "configs" / "gislr.aslex-sentences.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
import hashlib
CONFIG_SHA = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()

N_WORKERS = 8  # materialization threads (zlib releases the GIL)

DATA_DIR = gislr_dir()
OUT_DIR = CACHE_DIR / "gislr_aslex_sentences" / CONFIG["dataset_version"]
SEQ_DIR = OUT_DIR / "sequences"
PLAN_PATH = OUT_DIR / "plan.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"
SEQ_DIR.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, obj) -> None:
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj, indent=1), encoding="utf-8")
    os.replace(tmp, path)


print(f"source:  {DATA_DIR}  ({CONFIG['source_split']}.csv)")
print(f"output:  {OUT_DIR}")
print(f"config:  {CONFIG_PATH.name}  sha256={CONFIG_SHA[:12]}")


source:  C:\Users\user2\.cache\kagglehub\datasets\bracu23101281\gislr-stratified\versions\1  (test.csv)
output:  C:\Users\user2\repository\data\cache\gislr_aslex_sentences\v1
config:  gislr.aslex-sentences.json  sha256=3725119f2e9b


## 1. Filter the corpus to ASL-LEX-mapped glosses

Reuses `sb.recognize.recognition2.dataset.load_filtered_corpus` (imports `aslex.gloss_entries` and
`sequences.corpus.load_sentences`, both unmodified).

In [2]:
# ============================================================
# Filter: which glosses map into ASL-LEX, which corpus sentences survive
# ============================================================
label_map = load_label_map(DATA_DIR)
glosses = sorted(label_map.keys())
FILTERED = aslex_dataset.load_filtered_corpus(glosses, version=CONFIG["corpus_version"])

print(f"GISLR glosses: {len(glosses)}, ASL-LEX-mapped: {len(FILTERED['allowed_glosses'])}")
print(f"corpus sentences: {FILTERED['n_sentences_total']} total, "
      f"{FILTERED['n_sentences_kept']} kept (every gloss ASL-LEX-mapped), "
      f"{FILTERED['n_sentences_total'] - FILTERED['n_sentences_kept']} dropped")
print(f"glosses covered by surviving sentences: {len(FILTERED['glosses_covered'])} / "
      f"{len(FILTERED['allowed_glosses'])} mapped")
if FILTERED["glosses_lost"]:
    print(f"WARNING -- mapped but no surviving sentence uses them: {sorted(FILTERED['glosses_lost'])}")


GISLR glosses: 250, ASL-LEX-mapped: 233
corpus sentences: 1757 total, 1491 kept (every gloss ASL-LEX-mapped), 266 dropped
glosses covered by surviving sentences: 233 / 233 mapped


## 2. Plan: assign every ASL-LEX-mapped test clip to a sequence

Same planner as GISLR-Sentences v1 (`compose.plan_sentences`, unmodified), given the filtered sentence
list and a clip pool pre-restricted to ASL-LEX-mapped glosses (`filter_clip_pool`) so nothing is left as an
"orphan" for a reason this notebook already knows about (an unmapped-gloss clip).

In [3]:
# ============================================================
# Plan (cached in plan.json, keyed by config hash)
# ============================================================
if PLAN_PATH.exists() and json.loads(PLAN_PATH.read_text())["config_sha256"] == CONFIG_SHA:
    PLAN = json.loads(PLAN_PATH.read_text())
    print(f"plan loaded from {PLAN_PATH.name}")
else:
    src_df = pd.read_csv(DATA_DIR / f"{CONFIG['source_split']}.csv")
    src_df = aslex_dataset.filter_clip_pool(src_df, FILTERED["allowed_glosses"])
    rng = np.random.default_rng(CONFIG["seed"])
    synth = dict(gap_range=tuple(CONFIG["synth"]["gap_frames"]),
                 rest_range=tuple(CONFIG["synth"]["rest_frames"]))
    bar = tqdm(total=len(src_df), desc="plan sentence split")
    sent_plan, stats = compose.plan_sentences(
        src_df, FILTERED["sentences_kept"], rng, **synth, **CONFIG["planner"],
        progress=lambda done, total: bar.update(done - bar.n))
    bar.close()
    for i, r in enumerate(sent_plan):
        r["seq_id"] = f"s{i:05d}"
    PLAN = {"config_sha256": CONFIG_SHA, "stats": stats, "sequences": sent_plan}
    write_json(PLAN_PATH, PLAN)
    print(f"plan written to {PLAN_PATH.name}")

rows = PLAN["sequences"]
st = PLAN["stats"]
print(f"{len(rows)} sequences, {st['n_slots']} sign slots, {st['n_clips']} clips used, "
      f"reuse {st['reuse_rate']:.1%}, orphans {st['n_orphan_clips']}, "
      f"{st['n_distinct_sentences']} distinct sentences")


plan sentence split:   0%|          | 0/17614 [00:00<?, ?it/s]

plan written to plan.json
6335 sequences, 19286 sign slots, 17614 clips used, reuse 8.7%, orphans 4, 1060 distinct sentences


## 3. Materialize the sequences (resumable)

Same composer as GISLR-Sentences v1 (`compose.materialize`/`write_sequence`, unmodified). The label map is
GISLR's own 250-class `sign_to_prediction_index_map.json`, kept as-is (not re-derived to a 0–232 range) so
labels stay comparable with every other GISLR artifact in the repo.

In [4]:
# ============================================================
# Build every sequence npz (manifest-driven, resumable)
# ============================================================
src_df = pd.read_csv(DATA_DIR / f"{CONFIG['source_split']}.csv")
relpath_of = dict(zip(src_df["uid"], src_df["npz_relpath"]))
label_of = label_map
manifest = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else {}


def build(i_row):
    i, row = i_row
    rel = f"sequences/{row['kind']}/{row['seq_id']}.npz"
    try:
        arrays = compose.materialize(
            row, DATA_DIR, relpath_of, label_of,
            np.random.default_rng([CONFIG["seed"], i]),
            rest_jitter=CONFIG["synth"]["rest_jitter"])
        compose.write_sequence(OUT_DIR / rel, arrays)
        return row["seq_id"], {"status": "done",
                               **compose.summarize_row(row, arrays, row["seq_id"], rel)}
    except Exception as e:  # recorded, retried on the next run
        return row["seq_id"], {"status": "failed", "error": repr(e)}


todo = [(i, r) for i, r in enumerate(rows) if manifest.get(r["seq_id"], {}).get("status") != "done"]
bar = tqdm(total=len(rows), initial=len(rows) - len(todo), desc="materialize")
n_failed = 0
with ThreadPoolExecutor(N_WORKERS) as ex:
    for k, (seq_id, rec) in enumerate(ex.map(build, todo), 1):
        manifest[seq_id] = rec
        n_failed += rec["status"] == "failed"
        bar.update(1)
        bar.set_postfix(failed=n_failed)
        if k % 500 == 0:
            write_json(MANIFEST_PATH, manifest)
write_json(MANIFEST_PATH, manifest)
bar.close()
done = sum(v["status"] == "done" for v in manifest.values())
print(f"{done}/{len(rows)} done, {n_failed} failed this run")


materialize:   0%|          | 0/6335 [00:00<?, ?it/s]

6335/6335 done, 0 failed this run


## 4. Round-trip check

A seeded sample: every segment of a built sequence slices back to its source clip bit-for-bit
(`compose.check_roundtrip`, unmodified).

In [5]:
# ============================================================
# Round-trip sample
# ============================================================
from sb.recognize.sequences.compose import check_roundtrip, read_sequence

rng = np.random.default_rng(CONFIG["seed"] + 1)
sample = rng.choice([r for r in rows if manifest.get(r["seq_id"], {}).get("status") == "done"],
                     size=min(CONFIG["n_roundtrip_checks"], len(rows)), replace=False)
for row in tqdm(sample, desc="roundtrip"):
    rel = manifest[row["seq_id"]]["relpath"] if "relpath" in manifest[row["seq_id"]] else f"sequences/{row['kind']}/{row['seq_id']}.npz"
    arrays = read_sequence(OUT_DIR / rel)
    check_roundtrip(arrays, row, DATA_DIR, relpath_of)
print(f"{len(sample)}/{len(sample)} sequences round-tripped exactly")


roundtrip:   0%|          | 0/100 [00:00<?, ?it/s]

100/100 sequences round-tripped exactly
